# Import libs

In [70]:
# import tensorflow and keras
import tensorflow as tf
from tensorflow import keras
from keras import layers
import keras.backend as K

# import numpy to create arrays for trial runs
import numpy as np
# matplotlib for visualisations
import matplotlib.pyplot as plt
# if using Colab, for later saving of models and loading data
from google.colab import drive
drive.mount('/content/drive')
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from pathlib import Path
from glob import glob
from PIL import Image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [71]:
#!unzip -q /content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn.zip -d /content/drive/MyDrive/BMET5933/WEEK_10

In [72]:
# dataset path config
train_vali_dataset_path = Path("/content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn")

# get class names and create mapping to labels
class_names = sorted([folder.name for folder in train_vali_dataset_path.iterdir() if folder.is_dir()])
class_to_label = {class_name: idx for idx, class_name in enumerate(class_names)}
print("Class to label mapping:", class_to_label)


Class to label mapping: {'centromere': 0, 'coarse_speckled': 1, 'fine_speckled': 2, 'homogeneous': 3, 'nucleolar': 4}


In [73]:
SEED=42
BATCH_SIZE=32
IMAGE_DIR=str(train_vali_dataset_path) # this is the path to the directory containing the class subdirectories
RESCALED_IMAGE_SIZE=(51,51) # images will be all resized to this - they will have 3 channels

training_ds = keras.preprocessing.image_dataset_from_directory(
    IMAGE_DIR,
    batch_size=BATCH_SIZE,
    image_size=RESCALED_IMAGE_SIZE,
		shuffle=True,
		seed = SEED,
    validation_split=0.3,
    subset='training',
		color_mode = 'grayscale'
)

validation_ds = keras.preprocessing.image_dataset_from_directory(
	IMAGE_DIR,
	labels='inferred',
	label_mode='int',
	batch_size=BATCH_SIZE,
	image_size=RESCALED_IMAGE_SIZE,
	shuffle=True,
	seed = SEED,
	validation_split=0.3,
	subset='validation',
	color_mode = 'grayscale'
)

# you can optimise data loading with prefetching
PREFETCH_SIZE = tf.data.AUTOTUNE
training_ds = training_ds.prefetch(buffer_size=PREFETCH_SIZE)
validation_ds = validation_ds.prefetch(buffer_size=PREFETCH_SIZE)


Found 453 files belonging to 5 classes.
Using 318 files for training.
Found 453 files belonging to 5 classes.
Using 135 files for validation.


In [74]:
# define a shallow CNN model
def create_shallow_cnn(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(num_classes, activation='softmax'),

	])
	return model

# initialise model and print summary
shallow_cnn = create_shallow_cnn(input_shape=(51, 51, 1), num_classes=len(class_names))

# check model architecture
shallow_cnn.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_35 (Conv2D)              │ (None, 47, 47, 16)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_35 (MaxPooling2D) │ (None, 23, 23, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_17 (Flatten)            │ (None, 8464)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 5)              │        42,325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,741 (166.96 KB)

 Trainable params: 42,741 (166.96 KB)

 Non-trainable params: 0 (0.00 B)

In [78]:
# define a deeper CNN
def create_deeper_cnn(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(64, activation='relu'),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

# initialise deeper model and print summary
deeper_cnn = create_deeper_cnn(input_shape=(51, 51, 1), num_classes=len(class_names))

# check deeper model architecture
deeper_cnn.summary()

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_39 (Conv2D)              │ (None, 49, 49, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_39 (MaxPooling2D) │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_40 (Conv2D)              │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_40 (MaxPooling2D) │ (None, 11, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 9, 9, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_19 (Flatten)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,133 (875.52 KB)

 Trainable params: 224,133 (875.52 KB)

 Non-trainable params: 0 (0.00 B)

In [76]:
LEARNING_RATE = 1e-4 # Some common values are 1e-3 (0.001) or 1e-5 (0.00001)
BATCH_SIZE = 32 # train with this many images per iteration [5, 10, or 20 might be good for a trial run]
NUM_EPOCHS = 25 # how many epochs to train for (each epoch visits the training data once) [5 might be good for a trial run]

# 1. train shallow CNN
# compile the model
shallow_cnn.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# train_history will store the metrics for each epoch, for use in generating graphs
shallow_train = shallow_cnn.fit(
    training_ds,
    validation_data=validation_ds,
    epochs=NUM_EPOCHS
)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 279ms/step - accuracy: 0.2421 - loss: 12.9168 - val_accuracy: 0.2370 - val_loss: 6.8355
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 188ms/step - accuracy: 0.2736 - loss: 5.9156 - val_accuracy: 0.2222 - val_loss: 3.3789
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 210ms/step - accuracy: 0.2767 - loss: 3.4140 - val_accuracy: 0.3259 - val_loss: 3.2612
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 463ms/step - accuracy: 0.3333 - loss: 2.6543 - val_accuracy: 0.3185 - val_loss: 2.3164
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 520ms/step - accuracy: 0.3396 - loss: 2.2531 - val_accuracy: 0.3333 - val_loss: 2.2804
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 355ms/step - accuracy: 0.3616 - loss: 2.1495 - val_accuracy: 0.2815 - val_loss: 2.0179
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 218ms/step - accuracy: 0.3836 - loss: 1.8688 - val_accuracy: 0.3333 - val_loss: 2.0728
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 245ms/step - accuracy: 0.4057 - loss: 1.8293 - val_accuracy: 0

In [79]:
# 2. train deeper CNN
# config for deeper CNN
deeper_cnn.compile(
    optimizer = keras.optimizers.Adam(LEARNING_RATE),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)
# train deeper CNN
deeper_train = deeper_cnn.fit(
  training_ds,
	validation_data=validation_ds,
	epochs=NUM_EPOCHS
)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.2264 - loss: 2.7958 - val_accuracy: 0.3630 - val_loss: 2.0216
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 455ms/step - accuracy: 0.2767 - loss: 1.8301 - val_accuracy: 0.3407 - val_loss: 1.6721
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 261ms/step - accuracy: 0.3868 - loss: 1.5098 - val_accuracy: 0.4370 - val_loss: 1.3981
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 261ms/step - accuracy: 0.3931 - loss: 1.3595 - val_accuracy: 0.3259 - val_loss: 1.4572
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 251ms/step - accuracy: 0.4623 - loss: 1.2478 - val_accuracy: 0.4889 - val_loss: 1.2623
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 257ms/step - accuracy: 0.4717 - loss: 1.1835 - val_accuracy: 0.3778 - val_loss: 1.2583
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 391ms/step - accuracy: 0.5535 - loss: 1.0910 - val_accuracy: 0.5259 - val_loss: 1.1740
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 262ms/step - accuracy: 0.5975 - loss: 1.0401 - val_accuracy: 0.

In [91]:
test_dataset_path = Path('/content/drive/MyDrive/BMET5933/WEEK_10/hep2img_testset')

def read_img(img_path):
  # read image and convert into np array
  img = np.array(Image.open(img_path))

  # expand channel dim
  if img.ndim == 2:
    img = img[..., np.newaxis]
  img = img / 255.0
  img = tf.image.resize(img, (51, 51))
  return img

def model_predict(model, image, class_names: list):
  image = np.array(image)

  if image.ndim == 2:
    # expand channel dim and norm
    image = np.expand_dims(image, axis = 0)
    image = image / 255.0
    image = tf.image.resize(51, 51)

  # predict image label
  prediction = model.predict(image)
  label_num = np.argmax(prediction)
  label = class_names[label_num]
  return label

def




In [90]:
# get all folder name
class_list = sorted([folder.name for folder in test_dataset_path.iterdir() if folder.is_dir()])

# define a dict storing all test sample path
test_sample_path = {}
for class_name in class_list:
  class_folder_path = test_dataset_path / class_name
  class_samples = sorted(class_folder_path.glob('*.png'))
  test_sample_path[class_name] = class_samples



# evaluate both model on external test set
# 1. Evaluate testset on shallow model
img1 = read_img(test_sample_path['centromere'][0])

# expand for batch dim
img1 = np.expand_dims(img1, axis=0)
# predict on image1
shallow_prediction1 = model_predict(shallow_cnn, img1, class_list)
print(shallow_prediction1)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
centromere
